# System One Model

In [ ]:
import os

# Set this before initializing PyTorch CUDA operations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Training dataset class

In [ ]:
from pydantic import BaseModel, model_validator
from typing import List
from dataclasses import dataclass

@dataclass
class ToolCall:
  tool_name: str
  response: str

@dataclass
class StateAction:
  instruction: str
  enabled_tools: List[str]
  trajectory: List[ToolCall]
  action: str

  @model_validator(mode="after")
  def validate_action(self):
    if self.action not in self.enabled_tools:
      raise ValueError(
          f"'action' must be one of {self.enabled_tools}, got '{self.action}'"
      )

    return self

## Dataset and Dataloader

### Loading dataset from HF

In [7]:
from datasets import load_dataset

ds = load_dataset("ScaleAI/MCP-Atlas")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['TASK', 'ENABLED_TOOLS', 'PROMPT', 'GTFA_CLAIMS', 'TRAJECTORY'],
        num_rows: 500
    })
})


In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['TASK', 'ENABLED_TOOLS', 'PROMPT', 'GTFA_CLAIMS', 'TRAJECTORY'],
        num_rows: 500
    })
})

In [2]:
ds = ds['train']

### Validating loaded dataset structure

In [ ]:
def validate_trajectory_format(trajectory):
  n = len(trajectory)
  for i in range(n):
    if (i+1)%2:
      assert trajectory[i]['role'] == 'assistant'
    else:
      assert trajectory[i]['role'] == 'tool'

### Creating Custom Dataset for training

In [ ]:
import json
import torch

def prepare_input(ds):
  input_datasets = []

  for idx, data in enumerate(ds):
    # if idx >= 50:
    #   return
    prompt = data['PROMPT']
    trajectory = json.loads(data['TRAJECTORY'])
    enabled_tools = json.loads(data['ENABLED_TOOLS'])
    processed_enabled_tools = []
    #check if enabled_tools is dictionary
    #enabled tools: {'name': 'cli-mcp-server_run_command',
    #'optionalParams': [], 'requiredParams': ['command']}
    for enabled_tool in enabled_tools:
      if isinstance(enabled_tool, dict):
        processed_enabled_tools.append(enabled_tool['name'])
      elif isinstance(enabled_tool, str):
        processed_enabled_tools.append(enabled_tool)
      else:
        raise(f"Invalid input format for index: {idx}")

    try:
      validate_trajectory_format(trajectory)
    except Exception:
      continue

    assert isinstance(trajectory, list)
    for i in range(0, len(trajectory), 2):
      tool_calls = trajectory[i].get('tool_calls', [])
      if len(tool_calls):
        assert len(tool_calls) == 1
        tool = tool_calls[0]
      else:
        continue

      assert tool is not None

      history = trajectory[:i]

      processed_history = []
      for turn in history:
        if 'content' in turn and isinstance(turn['content'], str):
          turn['tool_calls'][0]['function']['arguments'] = ""
        else: #tool call
          # No need to add tools response
          continue
        processed_history.append(turn)
      # try:
      #   tool_name = json.loads(tool['function']['name'])
      #   print(f"{idx} has tool name {tool_name}")
      # except:
      #   print(f"Processed {idx}")
      state_action = StateAction(
          prompt,
          processed_enabled_tools,
          processed_history,
          tool['function']['name']
      )
      input_datasets.append(state_action)

  return input_datasets

#### Input Dataset class

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset

In [ ]:
class InputDataset(Dataset):
  def __init__(self, inputs):
    self.inputs = inputs
    # self.count = 0

  def _validate_tools(self, enabled_tools):
    return all([isinstance(tool, str) for tool in enabled_tools])

  def __len__(self):
    return len(self.inputs)

  def __getitem__(self, idx):
    data: StateAction  = self.inputs[idx]
    instruction = f"""
    Follow the instruction and the predict the next tool call from the list of
    available_tools based in the provided context.

    ## Instruction:
    {data.instruction}

    ## Available Tools:
    {data.enabled_tools}

    ## Context:
    {data.trajectory}

    Predict the next tool to call.
    """
    labels = [0]*len(data.enabled_tools)
    # for tool in data.enabled_tools:
    #   # print(tool)
    #   if isinstance(tool, dict):
    #     self.count += 1
    #     continue
    #   tools_set[tool] += 1
    value_error = False
    try:
      labels[data.enabled_tools.index(data.action)] = 1
    except ValueError:
      data.enabled_tools.append(data.action)
      labels.append(1)
    # validate data integrity
    assert sum(labels) == 1
    self._validate_tools(data.enabled_tools)

    return {
        "instruction": instruction,
        "actions": data.enabled_tools,
        "labels": labels
    }

### DataLoader

In [ ]:
import torch
def collate_fun(batch):
  # 1. Extract raw fields from the batch
  instructions = [item["instruction"] for item in batch]
  actions_list = [item["actions"] for item in batch]
  labels_list = [item["labels"] for item in batch]

  max_S = max(len(actions) for actions in actions_list)
  padded_acts = []
  padded_labels = []
  # print(actions_list, labels_list)
  for acts, lbls in zip(actions_list, labels_list):
    seq_len = len(acts)
    rem = max_S - seq_len
    # Use list concatenation (+) to create new padded lists safely
    padded_acts.append(list(acts) + [""] * rem)
    padded_labels.append(list(lbls) + [0] * rem)
  # print(len(instructions), len(actions_list), len(torch.tensor(labels_list)))
  return instructions, padded_acts, torch.tensor(padded_labels, dtype=torch.float32)

### Create dataloader for input subset

In [ ]:
from torch.utils.data import random_split

# Input Dataset
input_datasets = prepare_input(ds)

# Create object of Dataset for Dataloader
dt = InputDataset(input_datasets)

# Split sizes
train_size = int(0.8 * len(dt))
test_size = len(dt) - train_size

print(f"Train size: {train_size} & test size: {test_size}")

# Split
train_dataset, test_dataset = random_split(dt, [train_size, test_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          collate_fn=collate_fun)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                         collate_fn=collate_fun)

print(f"Train Loader size: {len(train_loader)} & \
        Test Loader size: {len(test_loader)}")

Train size: 1932 & test size: 484
Train Loader size: 61 &         Test Loader size: 16


In [ ]:
bad = [i for i, item in enumerate(train_dataset)
       if len(item["actions"]) != len(item["labels"])]
print(len(bad), bad[:10])
for i in bad[:3]:
    print(len(train_dataset[i]["actions"]), len(train_dataset[i]["labels"]))

0 []


## System One Model

In [ ]:
from transformers import AutoModel, AutoTokenizer
import gc

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class SystemOneModel(nn.Module):
  _model = None
  _tokenizer = None

  def __init__(self, model_id, max_length: int = 16, proj_dim: int = 5,
               normalize_base_hidden_state: bool = False,
               init_temperature: float = 0.07):
    super().__init__()
    self.model_name = model_id
    SystemOneModel._load_resources(model_id)
    self.model = SystemOneModel._model
    self.device = self.model.device
    self.normalize_base_hidden_state = normalize_base_hidden_state
    # Append MLP Layer
    hidden_dim = self.model.config.hidden_size
    print(f"Size of hidden_dim: {hidden_dim}")
    self.hidden_dim = hidden_dim
    dim=128
    def make_head(hidden_dim, proj_dim, dim=1024, p=0.1):
      return nn.Sequential(
          nn.LayerNorm(hidden_dim),            # tames outlier dims in LM features
          nn.Linear(hidden_dim, dim),
          nn.GELU(),
          nn.Dropout(p),
          nn.Linear(dim, proj_dim),
      )

    # in __init__
    self.proj_head_states  = make_head(hidden_dim, proj_dim).to(self.device,
                                                                dtype=torch.float32)
    self.proj_head_actions = make_head(hidden_dim, proj_dim).to(self.device,
                                                                dtype=torch.float32)
    # # Keep the MLP for states and actions separate
    # self.proj_head_states = nn.Sequential(
    #     nn.Linear(hidden_dim, dim),
    #     nn.GELU(),
    #     nn.Linear(dim, proj_dim)
    #     #KEEP float32 to ensure gradient update is proper
    # ).to(self.device, dtype=torch.float32)

    # self.proj_head_actions = nn.Sequential(
    #     nn.Linear(hidden_dim, dim),
    #     nn.GELU(),
    #     nn.Linear(dim, proj_dim)
    # ).to(self.device, dtype=torch.float32)
    # why log?
    self.logit_scale = nn.Parameter(torch.tensor(1.0 / init_temperature,
                                                 device=self.device).log())
    # pad_token=eos_token, How does this affects?
    self.tokenizer = SystemOneModel._tokenizer
    # Set tokenizer padding to left.
    # This is important as we will be using decoder only transformer
    # and use the last token embedding. With PAD token on right
    # the last token would be PAD token for many inputs
    self.tokenizer.padding_side = 'left'
    self.tokenizer.truncation_side = 'left' #why?
    self.max_length = max_length

  @classmethod
  def _load_resources(cls, model_name):
    if not cls._model:
      cls._model = AutoModel.from_pretrained(model_name,
                                           device_map='auto')
      # Freeze the model
      for param in cls._model.parameters():
        param.requires_grad = False
      cls._tokenizer = AutoTokenizer.from_pretrained(model_name)

  def _encode(self, X, projection_head):
    tokenized = self.tokenizer(X,
                               return_tensors='pt',
                               padding=True,
                               max_length=self.max_length,
                               truncation=True).to(self.device)
    # print(tokenized)
    with torch.no_grad():
      outputs = self.model(**tokenized)
      # 1. Extract last hidden state first: (B, seq_len, hidden_dim)
      # -> (B, hidden_dim)
      # cast frozen-backbone output to fp32 before it enters the fp32 head
      last_hidden = outputs.last_hidden_state[:, -1, :].float()

    if self.normalize_base_hidden_state:
      # Should we normalize last_hidden?
      norm_last_hidden = F.normalize(last_hidden, p=2, dim=-1)
      # 2. Pass through MLP head: (B, hidden_dim) -> (B, proj_dim)
      encoded = projection_head(norm_last_hidden)
    else:
      encoded = projection_head(last_hidden)
    # remove tokenizer from GPU to free up GPU space
    # del tokenized
    # del outputs
    # del norm_last_hidden
    # torch.cuda.empty_cache()

    return encoded

  def _similarity_score(self, states: torch.tensor, actions: torch.tensor):
    # Normalize the embeddings
    states = F.normalize(states, p=2, dim=-1).to(self.device) #(B x d_emb)
    actions = F.normalize(actions, p=2, dim=-1).to(self.device) #(B x seq_len x d_emb)

    # Unsqueeze states to allow dot product
    states = states.unsqueeze(1) #(B x 1 x d_emb)

    # Simlarity
    sim = (states * actions).sum(dim=-1) #(B x seq_len)
    assert torch.sum(torch.tensor(sim.shape) != \
                     torch.tensor([actions.shape[0], actions.shape[1]])) == 0

    # del states
    # del actions
    # torch.cuda.empty_cache()
    return sim

  def contrastive_loss(self, logits: torch.tensor, labels: torch.tensor, action_mask=None):
    # B, seq_len = actions.shape[0], actions.shape[1]
    # scores = self._similarity_score(states, actions) #(B x seq_len)
    # targets = torch.zeros(B, seq_len)
    # for i in range(B):
    #   targets[i][i] = 1
    labels = labels.to(self.device)
    target_indices = labels.argmax(dim=-1)
    if action_mask is not None:
      # Mask out padded empty actions so they don't influence softmax
      logits = logits.masked_fill(~action_mask, -1e4)
    criterion = nn.CrossEntropyLoss()
    # self._loss(scores, targets)
    # Apply dynamic temperature scaling (clamped for numerical stability)
    logit_scale = self.logit_scale.exp().clamp(max=100.0)
    scaled_logits = logits * logit_scale
    # tau = 0.07 #removed the hard-coded to check if
    # training imprvoes which is stalled
    loss = criterion(scaled_logits, target_indices)
    return loss

  def forward(self, states, actions):
    if isinstance(states, (list, tuple)):
      states = self._encode(states, self.proj_head_states)

    flat_actions = [
        a if a.strip() else self._tokenizer.pad_token
        for row in actions
        for a in row
    ]
    B, S = len(actions), len(actions[0])

    # FIX: Process flat_actions in small chunks (e.g., 32 at a time)
    # to prevent transformer activation memory explosion
    chunk_size = 32
    all_encoded = []

    for i in range(0, len(flat_actions), chunk_size):
      chunk = flat_actions[i : i + chunk_size]
      chunk_encoded = self._encode(chunk, self.proj_head_actions)
      all_encoded.append(chunk_encoded)

    # Concatenate all chunks back together and reshape to (B, S, E)
    action_embeddings = torch.cat(all_encoded, dim=0).view(B, S, -1)

    # Create mask for valid (non-empty) actions
    action_mask = torch.tensor(
        [[act != "" for act in sample_actions] for sample_actions in actions],
        dtype=torch.bool,
        device=self.device,
    )

    logits = self._similarity_score(states, action_embeddings)
    return logits, action_mask  # (B x seq_len)

  def unload_current_model(self):
    """Completely frees GPU memory occupied by the active model."""
    if self.model is not None:
      # Move model off GPU to CPU first (helps avoid dangling pointers)
      self.model.to("cpu")

      del self.model
      del self.tokenizer

      self.model = None
      self.tokenizer = None
      self.model_name = None

      # Force memory release
      gc.collect()
      if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


### Create System one model class

In [ ]:
def systemone_model(model_id='Qwen/Qwen2.5-0.5B', max_length=2048, proj_dim=16):
  return SystemOneModel(model_id, max_length, proj_dim)

In [ ]:
model = systemone_model(max_length=2048,
                        proj_dim=512)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Size of hidden_dim: 896


In [ ]:
model

SystemOneModel(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2Ro

### Loading the model in drive

In [ ]:
# 1. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
def save(epoch, model: SystemOneModel):
  # 2. Define your destination folder in Google Drive
  checkpoint_dir = f"/content/drive/MyDrive/model_checkpoints"
  os.makedirs(checkpoint_dir, exist_ok=True)
  checkpoint_path = os.path.join(checkpoint_dir, f"proj_heads_checkpoint_{epoch}.pt")

  # 3. Package the projection head state dicts (and optionally the optimizer)
  checkpoint = {
      "proj_head_states": model.proj_head_states.state_dict(),
      "proj_head_actions": model.proj_head_actions.state_dict(),
      'logit_scale': model.logit_scale.data,
      # Optional: include optimizer if you want to resume exact momentum/step counts
      # 'optimizer_state': optimizer.state_dict(),
  }

  # 4. Save to Drive
  torch.save(checkpoint, checkpoint_path)

In [ ]:
# save("test", model)

## Training & Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader):
  """Returns (mean loss per sample, top-1 accuracy) over the dataloader."""
  model.proj_head_states.eval()
  model.proj_head_actions.eval()
  total_loss, total_correct, n = 0.0, 0.0, 0
  for states, actions, labels in tqdm(dataloader, desc="Evaluating",
                                      unit="batch", leave=False):
    logits, action_mask = model.forward(states, actions)
    loss = model.contrastive_loss(logits, labels, action_mask)
    target = labels.to(model.device).argmax(dim=-1)
    bs = target.shape[0]
    total_loss += loss.item() * bs
    total_correct += (logits.argmax(dim=-1) == target).float().sum().item()
    n += bs
  model.proj_head_states.train()
  model.proj_head_actions.train()
  return total_loss / n, total_correct / n

In [ ]:
class UpdateMonitor:
  """Logs what optimizer.step() actually did to the weights.

  Update/ratio/<param>         ||delta_w|| / ||w|| per step (rule of thumb ~1e-3)
  Update/frac_changed/<param>  fraction of weights that changed at all
  Update/ratio_total, Update/frac_changed_total   same over all params
  Grad/norm/<param>            per-tensor gradient norm (after clipping)
  Grad/hist/*, Weights/hist/*  histograms every `hist_every` steps
  """
  def __init__(self, named_params, writer, hist_every=50):
    self.params = [(n.replace(".", "/"), p) for n, p in named_params if p.requires_grad]
    self.writer = writer
    self.hist_every = hist_every
    self._prev = {}

  @torch.no_grad()
  def before_step(self, step):
    for n, p in self.params:
      self._prev[n] = p.detach().clone()
      if p.grad is not None:
        g = p.grad.float()
        self.writer.add_scalar(f"Grad/norm/{n}", g.norm().item(), step)
        if step % self.hist_every == 0:
          self.writer.add_histogram(f"Grad/hist/{n}", g, step)

  @torch.no_grad()
  def after_step(self, step):
    sq_delta = sq_weight = 0.0
    changed_total = numel_total = 0
    for n, p in self.params:
      new, old = p.detach().float(), self._prev[n].float()
      delta = new - old
      d_norm, w_norm = delta.norm().item(), new.norm().item()
      changed = (delta != 0).sum().item()

      self.writer.add_scalar(f"Update/ratio/{n}", d_norm / (w_norm + 1e-12), step)
      self.writer.add_scalar(f"Update/frac_changed/{n}", changed / delta.numel(), step)
      if step % self.hist_every == 0:
        self.writer.add_histogram(f"Weights/hist/{n}", new, step)

      sq_delta += d_norm ** 2
      sq_weight += w_norm ** 2
      changed_total += changed
      numel_total += delta.numel()

    self.writer.add_scalar("Update/ratio_total",
                           (sq_delta ** 0.5) / (sq_weight ** 0.5 + 1e-12), step)
    self.writer.add_scalar("Update/frac_changed_total", changed_total / numel_total, step)

In [ ]:
import math

In [ ]:
def topk_accuracy(logits, target_indices, action_mask, ks=(1, 3)):
    """logits: (B, S) raw cosine sims, target_indices: (B,), action_mask: (B, S) bool."""
    masked = logits.masked_fill(~action_mask, float("-inf"))
    n_valid = action_mask.sum(dim=-1)                      # (B,)
    out = {}
    for k in ks:
        kk = min(k, masked.shape[1])
        topk = masked.topk(kk, dim=-1).indices             # (B, kk)
        hit = (topk == target_indices.unsqueeze(1)).any(dim=-1)
        out[k] = hit.float().mean().item()
    # Chance level for each k, so you can read the numbers properly
    out["chance_top1"] = (1.0 / n_valid.float()).mean().item()
    out["chance_top3"] = (3.0 / n_valid.float()).clamp(max=1.0).mean().item()
    return out

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torch.optim.lr_scheduler import LambdaLR

def train(epochs: int,
          dataloader: DataLoader,
          model: SystemOneModel,
          eval_dataloader: DataLoader=None,
          log_dir="runs/mcp_contrastive",
          warmup_steps: int = 20,
          min_lr_ratio: float = 0.05):
  writer = SummaryWriter(log_dir=log_dir)
  model._model.eval() # set the LLM backbone to eval mode
  model.proj_head_states.train()
  model.proj_head_actions.train() # Only set the Projection head to train mode

  head_params = (list(model.proj_head_states.parameters())
             + list(model.proj_head_actions.parameters()))

  optimizer = torch.optim.AdamW([
    {"params": head_params,            "weight_decay": 0.01},
    {"params": [model.logit_scale],    "weight_decay": 0.0},
  ], lr=1e-3)

  total_steps = epochs * len(dataloader)

  def lr_lambda(step):
    if step < warmup_steps:
      return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr_ratio + (1 - min_lr_ratio) * cosine

  scheduler = LambdaLR(optimizer, lr_lambda)
  monitor = UpdateMonitor(
      list(model.proj_head_states.named_parameters(prefix="states"))
      + list(model.proj_head_actions.named_parameters(prefix="actions")),
      writer)
  global_step = 0

  for epoch in range(epochs):
      total_loss = 0.0

      for idx, (states, actions, labels) in enumerate(
          tqdm(dataloader, desc="Loading batches", unit="batch")):
        try:
          optimizer.zero_grad()
          # 1. Forward pass
          logits, action_mask = model.forward(states, actions)
          # 2. Compute Loss
          loss = model.contrastive_loss(logits, labels, action_mask)

          # 3. Backward pass
          loss.backward()

          # 4. Track Gradient Norm of Projection Head
          grad_norm = torch.nn.utils.clip_grad_norm_(
                      head_params + [model.logit_scale], max_norm=1.0)

          # snapshot weights + log per-layer grad stats, then step, then log the real update
          monitor.before_step(global_step)
          writer.add_scalar("LR", optimizer.param_groups[0]["lr"], global_step)
          optimizer.step()
          monitor.after_step(global_step)

          scheduler.step()
          # 5. Compute Metrics for Logging
          with torch.no_grad():
            target_indices = labels.to(model.device).argmax(dim=-1)

            # Top-1 Accuracy
            preds = logits.argmax(dim=-1)
            # print("(labels, preds, target)",labels, preds, target_indices)
            # print(target_indices, preds)
            m = topk_accuracy(logits, target_indices, action_mask)
            top1_acc, top3_acc = m[1], m[3]
            # top1_acc = (preds == target_indices).float().mean().item()

            # Similarity Gap (Positive vs Negative Similarity)
            pos_sim = logits.gather(1, target_indices.unsqueeze(1)).squeeze(1)
            mean_pos_sim = pos_sim.mean().item()

            neg_mask = action_mask.clone()
            for b_i in range(neg_mask.shape[0]):
                neg_mask[b_i, target_indices[b_i]] = False

            mean_neg_sim = logits[neg_mask].mean().item() if neg_mask.sum() > 0 else 0.0
            sim_gap = mean_pos_sim - mean_neg_sim

            # del target_indices
            # del preds
            # torch.cuda.empty_cache()

          # Log to TensorBoard
          writer.add_scalar("Loss/train", loss.item(), global_step)
          writer.add_scalar("Accuracy/Top-1", top1_acc, global_step)
          writer.add_scalar("Accuracy/Top-3", top3_acc, global_step)
          writer.add_scalar("Similarity/Positive", mean_pos_sim, global_step)
          writer.add_scalar("Similarity/Negative", mean_neg_sim, global_step)
          writer.add_scalar("Similarity/Gap", sim_gap, global_step)
          writer.add_scalar("Gradients/proj_head_norm", grad_norm.item(), global_step)

          global_step += 1
          total_loss += loss.item()
          save("latest_new", model)
        except:
          print(f"skipping {idx} in epoch {epoch}")
      avg_epoch_loss = total_loss / len(dataloader)
      print(f"Epoch [{epoch+1}/{epochs}] | Avg Loss: {avg_epoch_loss:.4f}")
      if eval_dataloader is not None:
        eval_loss, eval_acc = evaluate(model, eval_dataloader)
        writer.add_scalar("Epoch/Loss_eval", eval_loss, epoch + 1)
        writer.add_scalar("Epoch/Top1_eval", eval_acc, epoch + 1)
        msg = f" | Eval Loss: {eval_loss:.4f} | Eval Top-1: {eval_acc:.3f}"
        print(msg)


  writer.close()
  print("Training Complete! TensorBoard logs saved.")

In [ ]:
def load_checkpoint(model, filepath: str="/content/drive/MyDrive/model_checkpoints/proj_heads_checkpoint_latest_new.pt"):
    """Loads the trainable components from a checkpoint file."""
    checkpoint = torch.load(filepath, map_location=model.device)
    model.proj_head_states.load_state_dict(checkpoint['proj_head_states'])
    model.proj_head_actions.load_state_dict(checkpoint['proj_head_actions'])
    model.logit_scale.data = checkpoint['logit_scale'].to(model.device)
    print(f"Checkpoint successfully loaded from {filepath}")

In [ ]:
load_checkpoint(model)

Checkpoint successfully loaded from /content/drive/MyDrive/model_checkpoints/proj_heads_checkpoint_latest_new.pt


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/mcp_contrastive

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 6505), started 0:14:01 ago. (Use '!kill 6505' to kill it.)

<IPython.core.display.Javascript object>

In [ ]:
len(train_loader)

61

In [ ]:
train(epochs=50, dataloader=train_loader, model=model,
      eval_dataloader=test_loader,
      log_dir="runs/mcp_contrastive")
save(101, model)
import os
import shutil
from google.colab import drive, runtime

# 1. Ensure Google Drive is mounted (if not already)

# 2. Define paths
local_log_dir = "runs/mcp_contrastive"
# Change this path to wherever you want to store them in your Drive
drive_log_dir = "/content/drive/MyDrive/model_checkpoints/mcp_contrastive_logs"

# 3. Copy local logs to Google Drive
if os.path.exists(local_log_dir):
  # Remove old folder in Drive if it exists to avoid conflicts, then copy fresh
  if os.path.exists(drive_log_dir):
    shutil.rmtree(drive_log_dir)

  shutil.copytree(local_log_dir, drive_log_dir)
  print(f"✅ Successfully saved TensorBoard logs to Google Drive: {drive_log_dir}")
else:
  print(f"⚠️ Local log directory '{local_log_dir}' not found.")

# 4. Terminate Colab session to free up compute units
print("Terminating Colab session...")
runtime.unassign()

Loading batches: 100%|██████████| 61/61 [02:02<00:00,  2.01s/batch]


Epoch [1/50] | Avg Loss: 2.1080


 | Eval Loss: 1.9397 | Eval Top-1: 0.252


Loading batches: 100%|██████████| 61/61 [02:04<00:00,  2.04s/batch]


Epoch [2/50] | Avg Loss: 2.0276


 | Eval Loss: 1.9689 | Eval Top-1: 0.285


... (140 outputs left out)

 | Eval Loss: 1.9705 | Eval Top-1: 0.415


Loading batches: 100%|██████████| 61/61 [02:05<00:00,  2.05s/batch]


Epoch [50/50] | Avg Loss: 0.9549


 | Eval Loss: 1.9881 | Eval Top-1: 0.417
Training Complete! TensorBoard logs saved.
✅ Successfully saved TensorBoard logs to Google Drive: /content/drive/MyDrive/model_checkpoints/mcp_contrastive_logs
Terminating Colab session...


##